# ViFeedback — Colab training

**This notebook contains no logic.** It clones the repository and calls the same
`vifeedback` CLI the laptop runs, so a Colab-only bug is impossible and every result lands in
the same `results/registry.csv` schema (docs/ROADMAP.md § 4).

## What belongs here, and what does not

The reference machine is an RTX 3050 Laptop with **4.29 GB** of VRAM. Measured and calculated
fits (see ADR-009):

| Model | Params | VRAM (AdamW, fp16, batch 32) | Runs locally? |
|---|---|---|---|
| `phobert-base` | 135M | **3.6 GB measured** | yes |
| `phobert-base-v2` | 135M | ~3.6 GB | yes |
| `visobert` | 97M | ~3.0 GB | yes |
| `xlm-roberta-base` | 277M | ~5.9 GB | no — unless embeddings are frozen (~3.6 GB) |
| `phobert-large` | 368M | ~7.9 GB | **no** |

So this notebook is for `phobert-large`, full-embedding `xlm-roberta-base`, and any sweep too
large for one evening on the laptop. **Everything else should run locally** — it is faster in
wall-clock terms (69 s/epoch, no session limits) and it keeps the artifact trail in one place.

## What must NOT run here

**Any latency benchmark.** Phase 6 measures CPU p95 on the documented reference machine; a
number produced on a Colab VM is not comparable and does not belong in the registry
(docs/EVALUATION_PROTOCOL.md § Latency harness).

## 1. Confirm the GPU

If this reports a T4 (16 GB) or better, `phobert-large` will fit. If it reports no GPU, set
*Runtime → Change runtime type → GPU* and re-run.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
import torch
print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f"{p.name}  {p.total_memory/1e9:.2f} GB  capability {p.major}.{p.minor}")

## 2. Get the repository

Two routes. Use **A** once the repo has a remote; use **B** meanwhile — the project is a local
git repo with no remote yet, so uploading a zip of the working tree is the honest fallback.

In [ ]:
# --- Route A: clone (preferred, once a remote exists) ---
REPO_URL = ""  # e.g. "https://github.com/<user>/ViFeedback-NLP-Service.git"

import os, pathlib

if REPO_URL:
    !git clone -q $REPO_URL /content/vifeedback
    os.chdir("/content/vifeedback")
    print("cloned:", pathlib.Path.cwd())
else:
    print("REPO_URL is empty — use Route B below.")

In [ ]:
# --- Route B: upload a zip of the repo (no remote yet) ---
# On the laptop, from the repo root:
#   git archive --format=zip -o vifeedback.zip HEAD
# then upload vifeedback.zip when prompted.
import os, pathlib

if not REPO_URL:
    from google.colab import files

    up = files.upload()
    name = next(iter(up))
    !mkdir -p /content/vifeedback && unzip -qo $name -d /content/vifeedback
    os.chdir("/content/vifeedback")
    print("unpacked:", pathlib.Path.cwd())
    print(sorted(p.name for p in pathlib.Path.cwd().iterdir())[:12])

## 3. Install

Colab already ships torch, so only the project and its lighter dependencies are installed.
`jdk4py` and `py_vncorenlp` are needed only if you train on a **segmented** variant (ADR-010).

In [ ]:
!pip install -q -e . 2>&1 | tail -3
!pip install -q transformers datasets huggingface-hub pyarrow typer pyyaml py-cpuinfo 2>&1 | tail -3

# Only if a segmented preprocessing variant is being trained:
NEED_SEGMENTATION = False
if NEED_SEGMENTATION:
    !pip install -q underthesea pyvi jdk4py py_vncorenlp 2>&1 | tail -3

import vifeedback

print("vifeedback", vifeedback.__version__)

## 4. Data

Fetches UIT-VSFC and asserts the official split sizes (11,426 / 1,583 / 3,166). The integrity
suite runs here too — if the upstream data ever shifts, this fails loudly rather than producing
numbers against different data.

In [ ]:
!python -m vifeedback.cli data fetch
!python -m pytest tests/data -q

In [ ]:
# Only for segmented conditions (Phase 3). Skip for the raw P0 condition.
if NEED_SEGMENTATION:
    !python -m vifeedback.cli data segmenters
    !python -m vifeedback.cli data variants --name seg_vncorenlp

## 5. Train

The same CLI the laptop calls. Keep each run **under 30 minutes** and let the loop checkpoint,
so a dropped session costs one run rather than an evening (risk R4).

`phobert-large` needs a smaller batch and gradient accumulation to hold the effective batch at 32
— the comparison against `phobert-base` is only meaningful if the effective batch matches.

In [ ]:
# PhoBERT-large — the reason this notebook exists.
!python -m vifeedback.cli train run \
    --task sentiment \
    --model phobert-large \
    --recipe base \
    --preprocessing raw \
    --seeds all \
    --epochs 4 \
    --lr 1e-5 \
    --batch-size 16 \
    --max-length 96 \
    --phase 4

In [ ]:
# XLM-R base with full (unfrozen) embeddings — the multilingual control.
# The frozen-embedding version fits locally; run that one on the laptop instead.
!python -m vifeedback.cli train run \
    --task sentiment \
    --model xlmr-base \
    --recipe base \
    --seeds all \
    --epochs 4 \
    --batch-size 32 \
    --phase 4

## 6. Bring the results home

Only `results/` needs to come back. Merging the registry rows into the laptop's
`results/registry.csv` is what keeps every claim traceable to a `run_id` regardless of which
machine produced it — and `env.json` records which machine that was.

In [ ]:
import shutil

from google.colab import files

shutil.make_archive("/content/colab_results", "zip", "results")
files.download("/content/colab_results.zip")

# On the laptop, unpack somewhere OTHER than results/ and merge by run_id. The archive's
# registry.csv may contain rows that were already committed before the clone, so appending
# it blind would duplicate them:
#
#   unzip -o colab_results.zip -d /tmp/colab
#   cp -rn /tmp/colab/runs/* results/runs/
#   python - <<'EOF'
#   import pandas as pd
#   a = pd.read_csv('results/registry.csv')
#   b = pd.read_csv('/tmp/colab/registry.csv')
#   out = pd.concat([a, b]).drop_duplicates(subset='run_id', keep='first')
#   out.to_csv('results/registry.csv', index=False)
#   print(f'{len(out) - len(a)} new rows merged')
#   EOF


In [ ]:
# Optional: persist to Drive instead, so an interrupted session loses nothing.
from google.colab import drive

drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/vifeedback
!cp -r results /content/drive/MyDrive/vifeedback/
print("copied to Drive")

## 7. Reporting the split honestly

Runs from this notebook carry a different `env.json` — different GPU, different driver, possibly
different library versions. That is fine for **accuracy** numbers, which are hardware-independent,
and disqualifying for **latency** numbers, which are not.

In the final results table, mark Colab-trained rows and state the GPU. A reviewer should never
have to guess which machine produced a number.